# Warmup Cache — Pre-generate Analysis Embeddings

Notebook này chạy `stock_analysis.pipeline()` cho **tất cả các cửa sổ 20 ngày** mà agent sẽ gặp trong quá trình training.

**Mục đích:** Pipeline tự cache kết quả (report, LLM response, embedding) theo hash. Chạy notebook này 1 lần trước khi train → training không cần gọi API nữa.

**Thời gian ước tính:** ~15-20s/window (LLM call). Với 4 stocks × ~1266 windows = ~5000+ calls. Tuy nhiên pipeline cache theo nội dung report, nên nhiều windows liền kề có cùng report → cache hit.

**Khuyến nghị:** Chạy từng stock một, có thể interrupt và resume (đã cache sẽ không gọi lại).

In [ ]:
import sys, os, logging, time
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

sys.path.insert(0, '.')

from src.features.preprocessor import load_csv, time_split
from stock_analysis import pipeline

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger(__name__)

# Load config
with open('configs/config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

WINDOW = cfg['env']['window']  # 20
MODEL = cfg['analysis']['model']
PATHS = cfg['data']['paths']
TRAIN_RATIO = cfg['split']['train_ratio']
VAL_RATIO = cfg['split']['val_ratio']

print(f'Window: {WINDOW}')
print(f'Model: {MODEL}')
print(f'Stocks: {PATHS}')

In [ ]:
def get_all_windows(csv_path: str, window: int = 20) -> list[tuple[str, datetime, datetime]]:
    """
    Tạo danh sách tất cả (stock_id, date_start, date_end) mà env sẽ gọi.
    Bao gồm train + val + test splits.
    """
    stock_id = Path(csv_path).stem
    df = load_csv(csv_path)
    
    windows = []
    # Env bắt đầu từ step=window, đi đến cuối
    for step in range(window, len(df)):
        start_idx = max(0, step - window + 1)
        date_end = pd.Timestamp(df['date'].iloc[step]).to_pydatetime()
        date_start = pd.Timestamp(df['date'].iloc[start_idx]).to_pydatetime()
        windows.append((stock_id, date_start, date_end))
    
    return windows

# Tổng hợp tất cả windows
all_windows = []
for p in PATHS:
    ws = get_all_windows(p, WINDOW)
    all_windows.extend(ws)
    print(f'{Path(p).stem}: {len(ws)} windows ({ws[0][1].date()} → {ws[-1][2].date()})')

print(f'\nTổng: {len(all_windows)} windows cần cache')

## Chạy Cache

Cell dưới đây chạy pipeline cho từng window. Nếu đã có cache → skip ngay (< 1ms).

Có thể interrupt bất cứ lúc nào — progress đã cache sẽ không mất.

In [ ]:
def warmup_stock(stock_id: str, csv_path: str, window: int = 20, skip_every: int = 1):
    """
    Cache tất cả windows cho 1 stock.
    
    Args:
        skip_every: Chỉ cache mỗi N windows (1 = tất cả, 5 = mỗi 5 ngày).
                    Dùng skip_every > 1 để tiết kiệm thời gian nếu cần.
    """
    df = load_csv(csv_path)
    total = len(df) - window
    cached = 0
    errors = 0
    start_t = time.time()
    
    print(f'\n{"="*60}')
    print(f'  Caching {stock_id}: {total} windows (skip_every={skip_every})')
    print(f'{"="*60}')
    
    for i, step in enumerate(range(window, len(df), skip_every)):
        start_idx = max(0, step - window + 1)
        date_end = pd.Timestamp(df['date'].iloc[step]).to_pydatetime()
        date_start = pd.Timestamp(df['date'].iloc[start_idx]).to_pydatetime()
        
        try:
            result = pipeline(
                model=MODEL,
                stock_id=stock_id,
                date_start=date_start,
                date_end=date_end,
            )
            cached += 1
            
            # Progress log mỗi 50 windows
            if cached % 50 == 0:
                elapsed = time.time() - start_t
                rate = cached / elapsed
                remaining = (total // skip_every - cached) / max(rate, 0.01)
                print(f'  [{stock_id}] {cached}/{total//skip_every} cached '
                      f'({cached*100//(total//skip_every)}%) | '
                      f'{rate:.1f} win/s | ETA: {remaining/60:.1f}m')
                
        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f'  [ERROR] step={step}, date={date_end.date()}: {e}')
            elif errors == 4:
                print(f'  ... (suppressing further errors)')
    
    elapsed = time.time() - start_t
    print(f'\n  ✓ {stock_id} done: {cached} cached, {errors} errors, {elapsed/60:.1f} min')
    return cached, errors

### Chạy từng stock (có thể chạy riêng từng cell)

In [ ]:
# VNM — Stable, chạy đầu tiên
warmup_stock('VNM', 'data/VNM.csv', WINDOW, skip_every=1)

In [ ]:
# FPT
warmup_stock('FPT', 'data/FPT.csv', WINDOW, skip_every=1)

In [ ]:
# VIC
warmup_stock('VIC', 'data/VIC.csv', WINDOW, skip_every=1)

In [ ]:
# HPG
warmup_stock('HPG', 'data/HPG.csv', WINDOW, skip_every=1)

## Kiểm tra Cache

Sau khi chạy xong, kiểm tra số file đã cache:

In [ ]:
from pathlib import Path

st_data = Path('stock_analysis/st_data')
for stock_dir in sorted(st_data.iterdir()):
    if not stock_dir.is_dir():
        continue
    reports = list((stock_dir / 'reports').glob('*.md')) if (stock_dir / 'reports').exists() else []
    responses = list((stock_dir / 'responses').glob('*.md')) if (stock_dir / 'responses').exists() else []
    embeddings = list((stock_dir / 'responses').glob('*.npy')) if (stock_dir / 'responses').exists() else []
    print(f'{stock_dir.name}: {len(reports)} reports | {len(responses)} responses | {len(embeddings)} embeddings')

## Xong!

Bây giờ có thể chạy `python train.py` — pipeline sẽ hit cache và không cần gọi LLM API nữa.

```bash
python train.py --episodes 500
```